In [18]:
import csv, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

LOG_ROOT = "/home/vscode/Desktop/logs"
APPROACHES = {
    "advanced":       "agentic-advanced-2-evaluation-run-gpt5.1",
    "basic":          "agentic-basic-2-evaluation-run-gpt5.1",
    "agent_baseline": "agentic-agent_baseline-2-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"advanced": "NullRepair", "basic": "SinglePrompt", "agent_baseline": "mini-SWE-agent"}

def collect_resolved_ids(log_root, approaches):
    resolved = {key: set() for key in approaches}
    for benchmark in sorted(os.listdir(log_root)):
        bp = os.path.join(log_root, benchmark)
        if not os.path.isdir(bp) or benchmark == "total":
            continue
        for key, subdir in approaches.items():
            mp = os.path.join(bp, subdir, "metrics.tsv")
            if not os.path.exists(mp):
                continue
            with open(mp, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f, delimiter="\t"):
                    if row.get("TARGET_ERROR_RESOLVED_WITHOUT_NEW_ERRORS", "false").lower() == "true":
                        resolved[key].add(f"{benchmark}/{row['ID']}")
    return resolved

resolved = collect_resolved_ids(LOG_ROOT, APPROACHES)
for key, ids in resolved.items():
    print(f"{APPROACH_LABELS[key]:20s}: {len(ids):4d} resolved")

adv, bas, abl = resolved["advanced"], resolved["basic"], resolved["agent_baseline"]

only_adv  = len(adv - bas - abl)
only_bas  = len(bas - adv - abl)
adv_bas   = len((adv & bas) - abl)
only_abl  = len(abl - adv - bas)
adv_abl   = len((adv & abl) - bas)
bas_abl   = len((bas & abl) - adv)
all_three = len(adv & bas & abl)

print(f"\nOnly NullRepair: {only_adv}, Only SinglePrompt: {only_bas}, Only mini-SWE-agent: {only_abl}")
print(f"Adv∩Bas: {adv_bas}, Adv∩Abl: {adv_abl}, Bas∩Abl: {bas_abl}, All: {all_three}")
print(f"Total unique: {len(adv | bas | abl)}")

# Overall Venn
fig, ax = plt.subplots(figsize=(8, 6))
v = venn3(subsets=(only_adv, only_bas, adv_bas, only_abl, adv_abl, bas_abl, all_three),
          set_labels=("NullRepair", "SinglePrompt", "mini-SWE-agent"), ax=ax, alpha=0.55)
for lbl in v.set_labels:
    if lbl: lbl.set_fontsize(12); lbl.set_fontweight("bold")
for sid in ("100","010","110","001","101","011","111"):
    lbl = v.get_label_by_id(sid)
    if lbl: lbl.set_fontsize(11)
#ax.set_title(f"Errors resolved without new errors (all projects)\nUnique errors across all approaches: {len(adv|bas|abl)}", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_resolved_no_new_errors.png", dpi=150)
print("\nSaved venn_resolved_no_new_errors.png")

# Per-project
benchmarks = sorted(b for b in os.listdir(LOG_ROOT) if os.path.isdir(os.path.join(LOG_ROOT, b)) and b != "total")
ncols = 4; nrows = -(-len(benchmarks) // ncols)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 5 * nrows))
axes_flat = axes.flatten()
for idx, bm in enumerate(benchmarks):
    adv_b = {e.split("/",1)[1] for e in adv if e.startswith(bm+"/")}
    bas_b = {e.split("/",1)[1] for e in bas if e.startswith(bm+"/")}
    abl_b = {e.split("/",1)[1] for e in abl if e.startswith(bm+"/")}
    ax = axes_flat[idx]
    total_b = len(adv_b | bas_b | abl_b)
    if total_b == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(bm, fontsize=10, fontweight="bold"); ax.axis("off"); continue
    only_a=len(adv_b-bas_b-abl_b); only_b=len(bas_b-adv_b-abl_b); ab=len((adv_b&bas_b)-abl_b)
    only_c=len(abl_b-adv_b-bas_b); ac=len((adv_b&abl_b)-bas_b); bc=len((bas_b&abl_b)-adv_b); abc=len(adv_b&bas_b&abl_b)
    vb = venn3(subsets=(only_a,only_b,ab,only_c,ac,bc,abc), set_labels=("Adv","Basic","AgentBL"), ax=ax, alpha=0.55)
    for lbl in vb.set_labels:
        if lbl: lbl.set_fontsize(8)
    for sid in ("100","010","110","001","101","011","111"):
        lbl = vb.get_label_by_id(sid)
        if lbl: lbl.set_fontsize(9)
    ax.set_title(f"{bm}  (n={total_b})", fontsize=10, fontweight="bold")
for idx in range(len(benchmarks), len(axes_flat)):
    axes_flat[idx].axis("off")
fig.suptitle("Per-project: Errors resolved without new errors", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("venn_resolved_per_project.png", dpi=150, bbox_inches="tight")
print("Saved venn_resolved_per_project.png")


NullRepair          :  696 resolved
SinglePrompt        :  777 resolved
mini-SWE-agent      :  853 resolved

Only NullRepair: 48, Only SinglePrompt: 72, Only mini-SWE-agent: 91
Adv∩Bas: 68, Adv∩Abl: 125, Bas∩Abl: 182, All: 455
Total unique: 1041

Saved venn_resolved_no_new_errors.png


/home/vscode/NullAwayAnnotatorBaseline/.venv/lib/python3.12/site-packages/matplotlib_venn/layout/venn3/pairwise.py:169: UserWarning: Bad circle positioning.
  warnings.warn("Bad circle positioning.")


Saved venn_resolved_per_project.png


In [ ]:
import csv, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from venn import venn

LOG_ROOT = "/home/vscode/Desktop/logs"
APPROACHES = {
    "advanced":       "agentic-advanced-2-evaluation-run-gpt5.1",
    "basic":          "agentic-basic-2-evaluation-run-gpt5.1",
    "agent_baseline": "agentic-agent_baseline-2-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"advanced": "NullRepair", "basic": "SinglePrompt", "agent_baseline": "mini-SWE-agent"}

def collect_resolved_ids(log_root, approaches):
    resolved = {key: set() for key in approaches}
    for benchmark in sorted(os.listdir(log_root)):
        bp = os.path.join(log_root, benchmark)
        if not os.path.isdir(bp) or benchmark == "total":
            continue
        for key, subdir in approaches.items():
            mp = os.path.join(bp, subdir, "metrics.tsv")
            if not os.path.exists(mp):
                continue
            with open(mp, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f, delimiter="\t"):
                    if row.get("TARGET_ERROR_RESOLVED_WITHOUT_NEW_ERRORS", "false").lower() == "true":
                        resolved[key].add(f"{benchmark}/{row['ID']}")
    return resolved

resolved = collect_resolved_ids(LOG_ROOT, APPROACHES)
for key, ids in resolved.items():
    print(f"{APPROACH_LABELS[key]:20s}: {len(ids):4d} resolved")

adv, bas, abl = resolved["advanced"], resolved["basic"], resolved["agent_baseline"]

only_adv  = len(adv - bas - abl)
only_bas  = len(bas - adv - abl)
adv_bas   = len((adv & bas) - abl)
only_abl  = len(abl - adv - bas)
adv_abl   = len((adv & abl) - bas)
bas_abl   = len((bas & abl) - adv)
all_three = len(adv & bas & abl)

print(f"\nOnly NullRepair: {only_adv}, Only SinglePrompt: {only_bas}, Only mini-SWE-agent: {only_abl}")
print(f"Adv∩Bas: {adv_bas}, Adv∩Abl: {adv_abl}, Bas∩Abl: {bas_abl}, All: {all_three}")
print(f"Total unique: {len(adv | bas | abl)}")

# Overall Venn
fig, ax = plt.subplots(figsize=(8, 6))
v = venn({"NullRepair": resolved["advanced"], "SinglePrompt": resolved["basic"], "mini-SWE-agent": resolved["agent_baseline"]},
           ax=ax, alpha=0.55)
#for lbl in v.set_labels:
#    if lbl: lbl.set_fontsize(12); lbl.set_fontweight("bold")
#for sid in ("100","010","110","001","101","011","111"):
#    lbl = v.get_label_by_id(sid)
 #   if lbl: lbl.set_fontsize(11)
#ax.set_title(f"Errors resolved without new errors (all projects)\nUnique errors across all approaches: {len(adv|bas|abl)}", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_resolved_no_new_errors.png", dpi=150)
print("\nSaved venn_resolved_no_new_errors.png")

# Per-project
benchmarks = sorted(b for b in os.listdir(LOG_ROOT) if os.path.isdir(os.path.join(LOG_ROOT, b)) and b != "total")
ncols = 4; nrows = -(-len(benchmarks) // ncols)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 5 * nrows))
axes_flat = axes.flatten()
for idx, bm in enumerate(benchmarks):
    adv_b = {e.split("/",1)[1] for e in adv if e.startswith(bm+"/")}
    bas_b = {e.split("/",1)[1] for e in bas if e.startswith(bm+"/")}
    abl_b = {e.split("/",1)[1] for e in abl if e.startswith(bm+"/")}
    ax = axes_flat[idx]
    total_b = len(adv_b | bas_b | abl_b)
    if total_b == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(bm, fontsize=10, fontweight="bold"); ax.axis("off"); continue
    only_a=len(adv_b-bas_b-abl_b); only_b=len(bas_b-adv_b-abl_b); ab=len((adv_b&bas_b)-abl_b)
    only_c=len(abl_b-adv_b-bas_b); ac=len((adv_b&abl_b)-bas_b); bc=len((bas_b&abl_b)-adv_b); abc=len(adv_b&bas_b&abl_b)
    vb = venn(subsets=(only_a,only_b,ab,only_c,ac,bc,abc), set_labels=("Adv","Basic","AgentBL"), ax=ax, alpha=0.55)
    for lbl in vb.set_labels:
        if lbl: lbl.set_fontsize(8)
    for sid in ("100","010","110","001","101","011","111"):
        lbl = vb.get_label_by_id(sid)
        if lbl: lbl.set_fontsize(9)
    ax.set_title(f"{bm}  (n={total_b})", fontsize=10, fontweight="bold")
for idx in range(len(benchmarks), len(axes_flat)):
    axes_flat[idx].axis("off")
fig.suptitle("Per-project: Errors resolved without new errors", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("venn_resolved_per_project.png", dpi=150, bbox_inches="tight")
print("Saved venn_resolved_per_project.png")
